<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/Phase_12_Ensemble_Models/12_RandomForest_Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 12: Non-Linear Classification (Random Forest)
## Step 1: Bypassing the Limits of Geometric Distance

In Phase 11, k-Nearest Neighbors mathematically proved that our anomalies and normal operations are heavily entangled. Distance-based algorithms failed to draw a clean boundary, resulting in either a 90% miss rate or severe alarm fatigue.

To navigate this 162-dimensional geometric overlap, we are deploying a **Random Forest Classifier**. Instead of relying on straight-line distance, Random Forest builds hundreds of hierarchical decision trees. Each tree looks at a random subset of our PCA components and attempts to isolate the anomalies using non-linear splits. By aggregating the votes of all these trees (Ensemble Learning), we can carve highly complex boundaries that distance-based models cannot see.

Because we already diagnosed severe class imbalance (1170 Passes vs. 83 Fails), we will utilize Random Forest's built-in `class_weight='balanced'` parameter. This mathematically penalizes the algorithm heavily for missing an anomaly, eliminating the need to synthetically generate data with SMOTE.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import time

print("⏳ Loading data and initializing Random Forest engine...")

# 1. Load the PCA artifact
df = pd.read_csv('secom_pca_95variance_ready.csv')
X = df.drop(columns=['Target'])
y = df['Target']

# 2. Execute Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Initialize Random Forest with cost-sensitive learning (class_weight='balanced')
rf_engine = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)

# 4. Train the model
start_time = time.time()
rf_engine.fit(X_train, y_train)
training_time = time.time() - start_time

# 5. Predict on unseen Test Data
y_pred_rf = rf_engine.predict(X_test)

print(f"✅ 100 Decision Trees built in {training_time:.4f} seconds.\n")
print("🏆 Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["Pass (-1)", "Fail (1)"], zero_division=0))

# 6. Generate Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(6, 5), facecolor='#161b22')
ax.set_facecolor('#161b22')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=["Predicted Pass", "Predicted Fail"],
            yticklabels=["Actual Pass", "Actual Fail"],
            annot_kws={"size": 16, "weight": "bold"}, ax=ax)

ax.set_title('Random Forest Confusion Matrix (Baseline)', color='white', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#8b949e', labelsize=11)
plt.tight_layout()
plt.show()

⏳ Loading data and initializing Random Forest engine...


FileNotFoundError: [Errno 2] No such file or directory: 'secom_pca_95variance_ready.csv'